# 06 — FishNet CV Colab main

Upload this notebook to Google Drive (e.g. `MyDrive/UH_CV/colab_main.ipynb`) and open with **Open in Colab**.

**One-time on Drive:**
- Put the FishNet dataset under `UH_CV/data/fishnet/` (`images/` + `labels/`)
- Put `validation_lengths.csv` under `UH_CV/data/annotations/`
- Put this repo on Drive **or** set `CLONE_REPO = True` below

Runs and outputs go to `MyDrive/UH_CV/runs/`.

In [ ]:
# --- edit these if needed ---
CLONE_REPO = False  # True: clone fresh copy into /content/fishnet_cv each session
REPO_DRIVE = "/content/drive/MyDrive/UH_CV/fishnet_cv"  # if repo lives on Drive
REPO_GITHUB = "https://github.com/YOUR_USER/fishnet_cv.git"  # used when CLONE_REPO=True

RUN = True
PIPELINE = "baseline"  # baseline | advanced
METHOD = "bbox"        # bbox | pca | skeleton
RUN_NAME = "baseline_bbox_v1"
SPLIT = "valid"
OVERWRITE = True
VISUALIZE = False

In [ ]:
import os
import subprocess
import sys
from pathlib import Path

from google.colab import drive

drive.mount("/content/drive")

if CLONE_REPO:
    repo = Path("/content/fishnet_cv")
    if not (repo / "src").is_dir():
        subprocess.run(["git", "clone", "--depth", "1", REPO_GITHUB, str(repo)], check=True)
else:
    repo = Path(REPO_DRIVE)
    if not (repo / "src").is_dir():
        raise FileNotFoundError(f"Repo not found: {repo}. Fix REPO_DRIVE or set CLONE_REPO=True.")

os.chdir(repo)
if str(repo) not in sys.path:
    sys.path.insert(0, str(repo))

subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-e", "."], check=True)
os.environ.setdefault("KMP_DUPLICATE_LIB_OK", "TRUE")

from src.colab_bootstrap import setup_notebook_environment
from src.config import get_config
from src.experiments import run_experiment
from src.utils import setup_logging

_, storage = setup_notebook_environment(mount_drive=False, repo_path=repo)
setup_logging()
cfg = get_config()

print("Repo:", repo)
print("Storage:", storage)
print("Dataset:", cfg.resolve_dataset_root())
print("Runs:", cfg.runs_root)

In [ ]:
if not RUN:
    print("RUN=False — set RUN=True to execute.")
else:
    result = run_experiment(
        pipeline=PIPELINE,
        method=METHOD,
        split=SPLIT,
        run_name=RUN_NAME,
        validation_set_only=True,
        overwrite=OVERWRITE,
        visualize=VISUALIZE,
        cfg=cfg,
    )
    print("Done:", result.run_dir)
    if result.metrics:
        print(f"MAE={result.metrics.mae_mm:.2f} mm  RMSE={result.metrics.rmse_mm:.2f} mm")

### Outputs

| File | Location |
|------|----------|
| Predictions | `UH_CV/runs/<run_name>/predictions.csv` |
| Metrics | `.../metrics.json`, `comparison.csv` |
| Registry | `UH_CV/runs/experiments.csv` |

For multi-run grids, use `notebooks/03_run_experiments.ipynb` from the repo.